# opp_xgc_forward vs fdr_avg — opponent-difficulty scoping (render-not-decide)

**This notebook DECIDES NOTHING.** It runs, reports, and visualizes the mean-features **step-2** question:
*is a dynamic, defence-specific opponent rating (`opp_xgc_forward` — the specific upcoming opponent's
strictly-prior rolling conceded-xG) a real improvement over FPL's crude static one-number-per-team
`fdr_avg`?* The verdict lives in the **commit message + specs**, not here — and it is **REFUTED**
(`docs/model-redesign-mean-features-plan.md` step-2): `opp_xgc_forward` ranks *below* `fdr_avg` on both
goals and assists, so it was not shipped. These figures show *why*.

Everything is computed by the shipped machinery: `dal.pipeline.load_opponent_map` (the sanctioned opponent
identity accessor) + `model.features.build.add_opponent_xgc_forward` (the team-grain roll, broadcast on the
opponent, coverage-filled). Walk-forward, fit `gw<t` / predict `gw==t`, conditional-on-appearance.

## Setup — enrich the mart with the opponent identity, materialize the two rival signals

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np

from dal.pipeline import load as load_mart
from dal.pipeline import load_opponent_map
from model.eval.metrics import block_bootstrap_ci, cell_spearman, grouped_spearman, has_rank_signal
from model.eval.walkforward import MIN_ROWS_PER_POS, WARMUP_GW
from model.features.build import add_opponent_xgc_forward
from model.terms.assists import AssistsModel
from model.terms.goals import GoalsModel

warnings.simplefilter("ignore")

# Design-system palette (validated categorical order): goals=blue, assists=orange; ink for text/axes.
C = {"goals": "#2a78d6", "assists": "#eb6834", "fdr": "#1baf7a", "opp": "#eda100"}
INK, MUTED, SURFACE = "#52514e", "#b7b6b0", "#fcfcfb"
plt.rcParams.update({"figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "axes.edgecolor": MUTED,
                     "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK,
                     "text.color": INK, "axes.grid": True, "grid.color": "#eceae3", "axes.axisbelow": True})

# The raw mart drops the opponent identity (like fixture_id); recover it via the sanctioned accessor and
# materialize opp_xgc_forward (team-grain roll of conceded-xG, broadcast keyed on the OPPONENT).
mart = load_mart().mart.copy().merge(load_opponent_map(), on=["player_id", "gw"], how="left")
mart = add_opponent_xgc_forward(mart, window=5)

BASE = {GoalsModel: ["xg_roll3", "xg_roll5", "xgi_roll3", "xgi_roll5", "minutes_roll3"],
        AssistsModel: ["xa_roll3", "xa_roll5", "xgi_roll3", "xgi_roll5", "minutes_roll3"]}


def fit_preds(model_cls, feats):
    """Walk-forward E[target] for a feature design, aligned to the model's population."""
    pop = model_cls.population(mart).copy()
    pop["pred"] = model_cls(variant="selected", feature_override=feats).fit(mart).predictions.to_numpy()
    return pop

## Figure 1 — do the two signals agree? The dynamic-vs-static gap

`fdr_avg` is a coarse 2–5 tier; `opp_xgc_forward` is a continuous rolling estimate of the *same* opponent's
leakiness. If the dynamic signal carried extra, usable structure it would spread **within** each fdr tier in
a way that tracks returns. The boxes show that spread; the title reports their rank agreement.

In [ ]:
ev = mart[(mart["minutes"] > 0) & (mart["gw"] > WARMUP_GW)].dropna(subset=["fdr_avg", "opp_xgc_forward"])
tiers = sorted(ev["fdr_avg"].unique())
data = [ev.loc[ev["fdr_avg"] == t, "opp_xgc_forward"].to_numpy() for t in tiers]
rho = cell_spearman(ev["fdr_avg"].to_numpy(), ev["opp_xgc_forward"].to_numpy())

fig, ax = plt.subplots(figsize=(7.2, 3.8))
bp = ax.boxplot(data, positions=tiers, widths=0.6, patch_artist=True, showfliers=False,
                medianprops={"color": INK, "linewidth": 1.5})
ax.set_xticks(tiers)  # only real tiers, no half-integer auto-ticks between boxes
for box in bp["boxes"]:
    box.set(facecolor=C["fdr"], alpha=0.45, edgecolor=C["fdr"], linewidth=1.5)
for w in bp["whiskers"] + bp["caps"]:
    w.set(color=MUTED)
ax.set_xlabel("fdr_avg (static difficulty tier)")
ax.set_ylabel("opp_xgc_forward\n(dynamic rolling conceded-xG)")
ax.set_title(f"Within each static tier, the dynamic signal spreads wide — "
             f"but they agree only ρ={rho:.2f}", color=INK, fontsize=11)
fig.tight_layout()

## Figure 2 — the shipping unit: paired per-(gw, position) Spearman delta, `opp_xgc` − `fdr`

The head-to-head. For each (gw, position) cell we form `ρ(selected + opp_xgc) − ρ(selected + fdr)` (fdr
removed from the first, opp_xgc from the second — one opponent feature each), then block-bootstrap the pooled
mean. **A point left of the dashed 0-line means the dynamic signal ranks *worse* than the crude tier.**

In [ ]:
def paired_delta(model_cls, hi_feats, lo_feats):
    t = model_cls.target
    pop = model_cls.population(mart).copy()
    pop["hi"] = model_cls(variant="selected", feature_override=hi_feats).fit(mart).predictions.to_numpy()
    pop["lo"] = model_cls(variant="selected", feature_override=lo_feats).fit(mart).predictions.to_numpy()
    d = pop[pop["gw"] > WARMUP_GW].dropna(subset=["hi", "lo", t])
    d = d[d["position"].isin(model_cls.fit_positions)]
    out = {}
    for pos, g in d.groupby("position"):
        vals = [cell_spearman(c["hi"].to_numpy(), c[t].to_numpy()) - cell_spearman(c["lo"].to_numpy(), c[t].to_numpy())
                for _, c in g.groupby("gw")
                if has_rank_signal(c, "hi", t, MIN_ROWS_PER_POS) and has_rank_signal(c, "lo", t, MIN_ROWS_PER_POS)]
        if len(vals) >= 4:
            lo, hi = block_bootstrap_ci(np.asarray(vals), seed=0)
            out[pos] = (float(np.mean(vals)), lo, hi, len(vals))
    return out


fig, ax = plt.subplots(figsize=(7.2, 4.0))
order, ylabels, yi = [], [], 0
for model_cls, name in [(GoalsModel, "goals"), (AssistsModel, "assists")]:
    res = paired_delta(model_cls, [*BASE[model_cls], "opp_xgc_forward"], [*BASE[model_cls], "fdr_avg"])
    for pos in ("DEF", "MID", "FWD", "GK"):
        if pos in res:
            mean, lo, hi, n = res[pos]
            ax.errorbar(mean, yi, xerr=[[mean - lo], [hi - mean]], fmt="o", color=C[name], ecolor=C[name],
                        elinewidth=2, capsize=3, markersize=7)
            ylabels.append(f"{name} · {pos} (n={n})")
            yi += 1
    yi += 0.5
ax.axvline(0, color=INK, linestyle="--", linewidth=1)
ax.set_yticks(range(len(ylabels)))
ax.set_yticklabels(ylabels)
ax.margins(y=0.06)  # headroom so the top/bottom markers are not clipped at the frame
ax.invert_yaxis()
ax.set_xlabel("pooled Spearman Δρ:  opp_xgc  −  fdr    (← opp_xgc worse | opp_xgc better →)")
ax.set_title("opp_xgc_forward ranks BELOW fdr_avg — every point left of 0", color=INK, fontsize=11)
from matplotlib.lines import Line2D
ax.legend(handles=[Line2D([0], [0], marker="o", color=C["goals"], label="goals", linestyle=""),
                   Line2D([0], [0], marker="o", color=C["assists"], label="assists", linestyle="")],
          loc="lower right", frameon=False)
fig.tight_layout()

## Figure 3 — ablation: is `fdr` subsumed, or does `opp_xgc` complement it?

Absolute pooled within-(gw, position) ρ for four designs. If `opp_xgc` were the sharper opponent signal,
`+opp_xgc` would clear `+fdr` and `+both` would clear `+fdr`. Instead `+opp_xgc` sits *below the
no-opponent base* (net-negative noise) and `+both` sits below `+fdr` — dropping opp_xgc costs ~0.

In [ ]:
def pooled_rho(model_cls, feats):
    t = model_cls.target
    pop = fit_preds(model_cls, feats)
    ev = pop[(pop["gw"] > WARMUP_GW) & pop["position"].isin(model_cls.fit_positions)].dropna(subset=["pred", t])
    return grouped_spearman(ev, "pred", t, ["gw", "position"], MIN_ROWS_PER_POS)


designs = ["base", "+fdr", "+opp_xgc", "+both"]
fig, ax = plt.subplots(figsize=(7.2, 3.8))
x = np.arange(len(designs))
w = 0.38
for i, (model_cls, name) in enumerate([(GoalsModel, "goals"), (AssistsModel, "assists")]):
    b = BASE[model_cls]
    feats = {"base": b, "+fdr": [*b, "fdr_avg"], "+opp_xgc": [*b, "opp_xgc_forward"],
             "+both": [*b, "fdr_avg", "opp_xgc_forward"]}
    vals = [pooled_rho(model_cls, feats[d]) for d in designs]
    bars = ax.bar(x + (i - 0.5) * w, vals, w, label=name, color=C[name], edgecolor=SURFACE, linewidth=2)
    ax.bar_label(bars, fmt="%.3f", padding=2, color=INK, fontsize=8)
    ax.axhline(vals[0], color=C[name], linestyle=":", linewidth=1, alpha=0.6)  # each model's no-opponent base
ax.set_xticks(x)
ax.set_xticklabels(designs)
ax.set_ylabel("pooled within-position Spearman ρ")
ax.set_title("Adding opp_xgc_forward does not clear +fdr — and falls below the base", color=INK, fontsize=11)
ax.legend(frameon=False, loc="lower left")
fig.tight_layout()

## Figure 4 — coverage map: where `opp_xgc_forward` is missing before the fill

The raw broadcast is NaN when an opponent has no single team-fixture that gw — an opponent playing a
**double gameweek** (its rows are DGW-excluded). Those holes (spikes below) are filled with the
strictly-prior league-average conceded-xG, so the scored population is **same-`n` as fdr-only** (no silent
row loss). Early GWs are pre-warmup (`gw ≤ 3`, not scored).

In [ ]:
# Reconstruct the PRE-fill NaN rate by rebuilding the raw broadcast (the materialized column is filled).
from model.features.build import broadcast

raw = mart[mart["minutes"] > 0].copy()

played = raw[raw["minutes"] > 0]
team = (played.groupby(["team_id", "gw"], as_index=False)["xgc"].mean().rename(columns={"xgc": "team_xgc"})
        .sort_values(["team_id", "gw"]))
team["roll"] = team.groupby("team_id")["team_xgc"].transform(lambda s: s.shift(1).rolling(5, min_periods=1).mean())
tf = team[["team_id", "gw", "roll"]].rename(columns={"team_id": "opponent_team_id"})
raw["opp_raw"] = broadcast(raw.assign(opponent_team_id=raw["opponent_team_id"].astype("float64")),
                           tf.assign(opponent_team_id=tf["opponent_team_id"].astype("float64")),
                           ["roll"], keys=("opponent_team_id", "gw"))["roll"].to_numpy()
cov = raw[raw["gw"] > WARMUP_GW].groupby("gw")["opp_raw"].apply(lambda s: 100.0 * s.isna().mean())

fig, ax = plt.subplots(figsize=(7.2, 3.4))
colors = [C["opp"] if v > 0.5 else MUTED for v in cov.to_numpy()]
ax.bar(cov.index, cov.to_numpy(), color=colors, edgecolor=SURFACE, linewidth=0.5)
ax.set_xlabel("gameweek")
ax.set_ylabel("opp_xgc_forward NaN %\n(scored rows, pre-fill)")
ax.set_title("Coverage holes are DGW-opponent gameweeks — filled with the league prior, not dropped",
             color=INK, fontsize=10)
fig.tight_layout()

## Read-out (no decision)

- Figure 1: the two signals agree only weakly — the dynamic spread within a tier is large, but…
- Figure 2: …that spread does **not** help — the paired `opp_xgc − fdr` delta is *negative* at every cell.
- Figure 3: `+opp_xgc` falls below the no-opponent base; `+both` falls below `+fdr`; dropping opp_xgc costs ~0.
- Figure 4: coverage is a handful of DGW-opponent gameweeks, filled (not dropped) → same-`n` as fdr-only.

**The decision** (keep `fdr_avg`, do not ship `opp_xgc_forward`) is recorded in the commit message,
`model/terms/{goals,assists}/spec.py`, and `docs/model-redesign-mean-features-plan.md` — not here.